# ADK — Intermediate Agent

**Framework:** Google Agent Development Kit (ADK)  
**Level:** 02 — Intermediate  
**Model:** `gemini-2.0-flash`

### What's New vs Simple
| Feature | Simple | Intermediate |
|---|---|---|
| Tools | 2 | **3** (added `get_travel_advisory`) |
| Memory | None (single-turn) | **Multi-turn** (same session persists) |
| Output | Free text | **Structured** (`output_schema` → Pydantic) |
| Turns | 1 | **Multiple** — agent remembers the conversation |

## Concept: Multi-Turn Memory + Structured Output

```
Turn 1: "Briefing for Tokyo"          Turn 2: "Now Bangalore"         Turn 3: "Which is safer?"
         │                                     │                               │
         ▼                                     ▼                               ▼
┌──────────────────┐               ┌──────────────────┐            ┌──────────────────┐
│  Session State   │               │  Session State   │            │  Session State   │
│  [Turn 1 msgs]   │ ──────────▶   │  [Turn 1 msgs]   │ ─────────▶ │  [Turn 1 msgs]   │
│                  │               │  [Turn 2 msgs]   │            │  [Turn 1 msgs]   │
└────────┬─────────┘               └────────┬─────────┘            │  [Turn 2 msgs]   │
         │ uses tools                       │ uses tools            │  [Turn 3 msgs]   │
         ▼                                 ▼                        └──────────────────┘
   TravelBriefing{}                  TravelBriefing{}           Agent recalls Tokyo & Bangalore!
```

**Key ADK concept: `output_schema`**  
Set `output_schema=TravelBriefing` on the Agent → the model's final response is constrained to match the Pydantic schema. You get structured JSON back, not free text.

**Key ADK concept: persistent session**  
Reusing the same `session_id` across multiple `runner.run_async()` calls gives the agent memory of the full conversation history.

## Setup

In [ ]:
import asyncio
import os
from pydantic import BaseModel, Field
from dotenv import load_dotenv

from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner
from google.adk.sessions import InMemorySessionService
from google.genai import types as genai_types

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY in your .env"
print("✓ Ready")

## Structured Output Schema

Defining a Pydantic model and passing it as `output_schema` forces the agent to return structured data instead of free-form text.

In [ ]:
class TravelBriefing(BaseModel):
    city: str = Field(description="Name of the city")
    weather_summary: str = Field(description="Weather conditions and temperature")
    local_time: str = Field(description="Current local time with timezone")
    travel_advisory: str = Field(description="Safety advisory level and key notes")
    recommendation: str = Field(description="One-sentence travel recommendation")

print("Schema fields:", list(TravelBriefing.model_fields.keys()))

## Tools — 3 this time

In [ ]:
def get_weather(city: str) -> dict:
    """Get weather for a city.
    Args:
        city: City name.
    Returns:
        Weather dict.
    """
    data = {
        "london":    {"condition": "Cloudy",       "temp_c": 12, "humidity": 78},
        "new york":  {"condition": "Sunny",         "temp_c": 22, "humidity": 45},
        "bangalore": {"condition": "Rainy",         "temp_c": 25, "humidity": 85},
        "tokyo":     {"condition": "Clear",         "temp_c": 18, "humidity": 60},
        "paris":     {"condition": "Partly Cloudy", "temp_c": 16, "humidity": 65},
    }
    key = city.lower()
    if key in data:
        d = data[key]
        return {"city": city, "condition": d["condition"],
                "temperature_celsius": d["temp_c"], "humidity_percent": d["humidity"]}
    return {"error": f"No data for '{city}'."}


def get_time(city: str) -> dict:
    """Get local time for a city.
    Args:
        city: City name.
    Returns:
        Local time dict.
    """
    times = {
        "london": "14:30 GMT", "new york": "09:30 EST",
        "bangalore": "20:00 IST", "tokyo": "22:30 JST", "paris": "15:30 CET",
    }
    key = city.lower()
    if key in times:
        return {"city": city, "local_time": times[key]}
    return {"error": f"No time data for '{city}'."}


def get_travel_advisory(city: str) -> dict:
    """Get travel safety advisory for a city.
    Args:
        city: City name.
    Returns:
        Advisory level and notes.
    """
    advisories = {
        "london":    {"level": "Low",    "notes": "Routine precautions. Pickpocketing in tourist areas."},
        "new york":  {"level": "Low",    "notes": "Normal precautions. Avoid isolated areas at night."},
        "bangalore": {"level": "Medium", "notes": "Monsoon season affects transport."},
        "tokyo":     {"level": "Low",    "notes": "Very safe. Earthquake preparedness recommended."},
        "paris":     {"level": "Low",    "notes": "Routine precautions. Alert in crowded spots."},
    }
    key = city.lower()
    if key in advisories:
        a = advisories[key]
        return {"city": city, "advisory_level": a["level"], "notes": a["notes"]}
    return {"error": f"No advisory data for '{city}'."}


print("3 tools defined: get_weather, get_time, get_travel_advisory")

## Agent with output_schema

In [ ]:
agent = Agent(
    name="travel_briefing_assistant",
    model="gemini-2.0-flash",
    description="Creates comprehensive travel briefings.",
    instruction="""You are a professional travel intelligence assistant.
When asked about a city, always check weather, local time, AND travel advisory.
Build a complete picture using all three tools, then synthesize into a structured briefing.
Remember context from earlier in the conversation.""",
    tools=[get_weather, get_time, get_travel_advisory],
    output_schema=TravelBriefing,  # ← NEW: forces structured JSON output
)

print(f"Agent '{agent.name}' — output_schema: TravelBriefing")

## Multi-Turn Conversation

The key: **reuse the same `session_id`** across calls. The `InMemorySessionService` holds the full conversation history, so the agent remembers Turn 1 when answering Turn 2.

In [ ]:
session_service = InMemorySessionService()

async def run(query: str, session_id: str) -> str:
    runner = InMemoryRunner(agent=agent, session_service=session_service)
    response_text = ""
    async for event in runner.run_async(
        user_id="user_01",
        session_id=session_id,
        new_message=genai_types.Content(
            role="user", parts=[genai_types.Part(text=query)]
        ),
    ):
        if event.is_final_response() and event.content:
            for part in event.content.parts:
                if part.text:
                    response_text += part.text
    return response_text


# Create ONE session — reuse for all turns
session = await session_service.create_session(app_name="travel_app", user_id="user_01")
SESSION_ID = session.id
print(f"Session created: {SESSION_ID}")

In [ ]:
# Turn 1
query1 = "Give me a full travel briefing for Tokyo."
print(f"Turn 1 — User: {query1}")
r1 = await run(query1, SESSION_ID)
print(f"Agent (raw JSON):\n{r1}")

In [ ]:
import json

# Parse the structured output
try:
    briefing = TravelBriefing.model_validate_json(r1)
    print(f"City:           {briefing.city}")
    print(f"Weather:        {briefing.weather_summary}")
    print(f"Time:           {briefing.local_time}")
    print(f"Advisory:       {briefing.travel_advisory}")
    print(f"Recommendation: {briefing.recommendation}")
except Exception as e:
    print(f"Note: structured parse failed ({e}). Raw output above is still valid.")

In [ ]:
# Turn 2 — same session, agent has Tokyo context
query2 = "Now do the same for Bangalore. I prefer warm weather — which would you recommend?"
print(f"Turn 2 — User: {query2}")
r2 = await run(query2, SESSION_ID)
print(f"Agent: {r2}")

In [ ]:
# Turn 3 — agent should recall both cities from conversation history
query3 = "Based on what you told me, what's the safer city to visit?"
print(f"Turn 3 — User: {query3}")
r3 = await run(query3, SESSION_ID)
print(f"Agent: {r3}")

## Key Takeaways

| Feature | How ADK Does It | Why It Matters |
|---|---|---|
| **3 tools** | Add to `tools=[]` list | Agent selects which to call based on query |
| **Structured output** | `output_schema=TravelBriefing` | Guarantees parseable JSON from the LLM |
| **Multi-turn memory** | Same `session_id` across runs | Full conversation history preserved in `InMemorySessionService` |
| **Cross-turn reasoning** | Turn 3 references Turn 1 + 2 data | Agent reads the full session context window |

### Structured Output Tradeoff
When `output_schema` is set, the agent **cannot use tools on that final turn** — the response must conform to the schema. For tool-heavy workflows, use output_schema on a separate formatting step.

### What Changes at 03-Complex
- **Planning step**: agent explicitly plans before calling tools
- **Reflexion**: agent critiques its own output and rewrites if quality < threshold
- **Streaming**: display intermediate events (tool calls, partial text) in real-time

### Next: [03-Complex →](../03-complex/agent.ipynb)